In [1]:
import pandas as pd

df = pd.read_parquet(
    "../data/processed/orders_clean.parquet"
)

print(df.shape)

(4048, 31)


In [2]:
df["hubName"].value_counts()

hubName
Instant Foods(Noida)             824
Cross Line Events (EDelhi)       565
Rapid Enterprises                504
Instant Foods (SED)              399
Crossline Events (Noida)         390
Instant Foods (GZB)              381
Crossline Events (Meerut)        368
Cross Line Events (Ghaziabad)    347
NB Enterprises (West Delhi)      270
Name: count, dtype: int64

In [3]:
region_trends = (
    df.groupby(
        [
            "hubId",
            "hubName",
            "shopType",
            "skuNumber"
        ]
    )
    .agg(
        total_qty=(
            "effective_qty",
            "sum"
        ),

        unique_buyers=(
            "customerId",
            "nunique"
        ),

        total_orders=(
            "orderNumber",
            "nunique"
        )
    )
    .reset_index()
)

region_trends.head()

,hubId,hubName,shopType,skuNumber,total_qty,unique_buyers,total_orders
0,HB002,Instant Foods(Noida),General A,SKU00001,1,1,1
1,HB002,Instant Foods(Noida),General A,SKU00009,1,1,1
2,HB002,Instant Foods(Noida),General A,SKU00010,13,7,7
3,HB002,Instant Foods(Noida),General A,SKU00022,1,1,1
4,HB002,Instant Foods(Noida),General A,SKU00024,1,1,1


In [4]:
sku_lookup = (
    df[
        [
            "skuNumber",
            "itemName",
            "brand",
            "category"
        ]
    ]
    .drop_duplicates()
)

region_trends = (
    region_trends.merge(
        sku_lookup,
        on="skuNumber",
        how="left"
    )
)

region_trends.head()

,hubId,hubName,shopType,skuNumber,total_qty,unique_buyers,total_orders,itemName,brand,category
0,HB002,Instant Foods(Noida),General A,SKU00001,1,1,1,Chingles Filz Jar,DS Group,Confectionery
1,HB002,Instant Foods(Noida),General A,SKU00009,1,1,1,Chingles Maxi Tutti Fruiti Jar,DS Group,Confectionery
2,HB002,Instant Foods(Noida),General A,SKU00010,13,7,7,Pass Pass Meetha Magic 11.75g Hanger,Pass Pass,Mouth Fresheners
3,HB002,Instant Foods(Noida),General A,SKU00022,1,1,1,Pulse Kachha Aam 175 Candy,DS Group,Confectionery
4,HB002,Instant Foods(Noida),General A,SKU00024,1,1,1,Pulse Kachha Aam Jar (250 Candy),DS Group,Confectionery


In [5]:
region_trends[
    "region_rank"
] = (
    region_trends
    .groupby(
        [
            "hubId",
            "shopType"
        ]
    )
    ["unique_buyers"]
    .rank(
        ascending=False,
        method="dense"
    )
)

region_trends.head()

,hubId,hubName,shopType,skuNumber,total_qty,unique_buyers,total_orders,itemName,brand,category,region_rank
0,HB002,Instant Foods(Noida),General A,SKU00001,1,1,1,Chingles Filz Jar,DS Group,Confectionery,7.0
1,HB002,Instant Foods(Noida),General A,SKU00009,1,1,1,Chingles Maxi Tutti Fruiti Jar,DS Group,Confectionery,7.0
2,HB002,Instant Foods(Noida),General A,SKU00010,13,7,7,Pass Pass Meetha Magic 11.75g Hanger,Pass Pass,Mouth Fresheners,1.0
3,HB002,Instant Foods(Noida),General A,SKU00022,1,1,1,Pulse Kachha Aam 175 Candy,DS Group,Confectionery,7.0
4,HB002,Instant Foods(Noida),General A,SKU00024,1,1,1,Pulse Kachha Aam Jar (250 Candy),DS Group,Confectionery,7.0


In [7]:
(
    region_trends[
        region_trends["hubName"]
        == "Instant Foods(Noida)"
    ]
    .sort_values(
        "region_rank"
    )
    [
        [
            "itemName",
            "unique_buyers",
            "region_rank"
        ]
    ]
    .head(10)
)

,itemName,unique_buyers,region_rank
2,Pass Pass Meetha Magic 11.75g Hanger,7,1.0
51,Pass Pass Meetha Magic 11.75g Hanger,7,1.0
206,RG Pearl Elaichi MP Pouch 0.14g,23,1.0
123,Haldiram - Aloo Bhujia (24 pcs),8,1.0
117,Haldiram - Punjabi Tadka (24 pcs),8,1.0
287,Rajnigandha 4g,6,1.0
175,Rajnigandha 17g Zipper IC,5,1.0
13,Pass Pass Meetha Magic 2.2g (100 pcs),7,1.0
290,TZ 00 (with Silver) 0.45g,5,2.0
215,Rajnigandha 17g Zipper IC,18,2.0


In [8]:
region_trends.to_parquet(
    "../data/processed/region_trends.parquet",
    index=False
)

print("Saved")

Saved


In [9]:
print(region_trends.shape)

print(
    region_trends["hubName"]
    .nunique()
)

print(
    region_trends["shopType"]
    .nunique()
)

(1711, 11)
9
10


In [10]:
region_trends.shape

(1711, 11)

In [11]:
region_trends.sort_values(
    "region_rank"
)[
    [
        "hubName",
        "shopType",
        "itemName",
        "region_rank"
    ]
].head(15)

,hubName,shopType,itemName,region_rank
1702,Rapid Enterprises,Paan C,Act II Butter Popcorn,1.0
1664,Rapid Enterprises,Paan B,Rajnigandha 4g,1.0
1705,Rapid Enterprises,Paan C,Haldiram - Moong Dal (â‚¹5),1.0
1706,Rapid Enterprises,Paan C,Haldiram - Salted Peanuts (â‚¹5),1.0
1707,Rapid Enterprises,Paan C,Haldiram - Nut Cracker (â‚¹5),1.0
1708,Rapid Enterprises,Paan C,Haldiram - Aloo Bhujia (â‚¹5),1.0
2,Instant Foods(Noida),General A,Pass Pass Meetha Magic 11.75g Hanger,1.0
1709,Rapid Enterprises,Paan C,Rajnigandha 17g Zipper 1 Pcs,1.0
1703,Rapid Enterprises,Paan C,Haldiram - Bhujia (â‚¹5),1.0
1478,Cross Line Events (EDelhi),Paan C,Jabsons Hing Jeera Peanuts,1.0
